In [0]:
from pyspark.sql.functions import *

prediction_df = spark.read.table(
    "finops_catalog_2.finops.ml_predictions"
)

Alert

In [0]:
alerts_df = prediction_df.withColumn(
    "alert_type",
    when(col("pred_anomaly") == 1, "ML_ANOMALY")
    .when(col("cost_budget_pct")> 4,"Budget_ANOMALY")
    .when(col("high_variance_flag") == 1, "HIGH_VARIANCE")
    .when(col("cost_variance_7d_pct") > 0.3300, "COST_SPIKE_7D")
    .when(col("cost_variance_30d_pct") > 0.3340, "COST_SPIKE_30D")
    .otherwise("NORMAL")
)
display(alerts_df.limit(5))

Filter alerts

In [0]:
alerts_df = alerts_df.filter(
    col("alert_type") != "NORMAL"
)

Alert time

In [0]:
alerts_df = alerts_df.withColumn(
    "alert_time",
    current_timestamp()
)

In [0]:
alerts_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "finops_catalog_2.finops.cost_alerts"
    )

In [0]:
display(
    spark.read.table(
        "finops_catalog_2.finops.cost_alerts"
    ).limit(5)
)

Alert Summary

In [0]:
alert_summary = (
    alerts_df.groupBy("alert_type")
    .count()
    .orderBy(col("count").desc())
)

alert_summary.show()

Notebook Alart

In [0]:
from pyspark.sql.functions import col
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

def send_finops_alert(
    alerts_df,
    sender_email,
    app_password,
    receiver_email
):

    pdf = (
        alerts_df.groupBy("alert_type")
        .count()
        .orderBy(col("count").desc())
        .toPandas()
    )

    html_table = pdf.to_html(
        index=False,
        border=1
    )

    msg = MIMEMultipart("alternative")

    msg["Subject"] = "🚨 Cloud Cost Anomaly Alert"
    msg["From"] = sender_email
    msg["To"] = receiver_email

    html_body = f"""
    <html>
    <body>
        <h2>Cloud Cost Alert Summary</h2>
        {html_table}
        <br>
        Generated from Databricks FinOps Pipeline
    </body>
    </html>
    """

    msg.attach(MIMEText(html_body, "html"))

    server = smtplib.SMTP("smtp.gmail.com", 587)
    server.starttls()

    server.login(
        sender_email,
        app_password
    )

    server.sendmail(
        sender_email,
        receiver_email,
        msg.as_string()
    )

    server.quit()

    print("✅ Email Sent")

In [0]:
send_finops_alert(
    alerts_df,
    sender_email="omkarmhetre777@gmail.com",
    app_password="oskqnwzfjhzmjatq",
    receiver_email="omkarmhetre800@gmail.com"
)

![](/Workspace/Users/omkarmhetre800@gmail.com/Cloud-Cost-Anomaly-Detection/data/Screenshot 2026-06-15 100559.png)

![](path)